# 🎙️ Dorina TTS - Sıfırdan Türkçe Kadın Sesi Eğitimi (Kaggle)
Bu notebook, **Dorina** için sıfırdan FastSpeech 2 tabanlı Türkçe ses modelini eğitir.

**Gereken Ayar:** Sağ üstten `Settings -> Accelerator -> GPU T4 x2` veya `P100` seçin.

In [ ]:
# 1. Gerekli Kütüphanelerin Kurulumu
!pip install -q torchaudio librosa soundfile pyarrow pandas

In [ ]:
# 2. Türkçe Karakter Tokenizer Tanımı
import re
import os
import io
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import soundfile as sf
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display

SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SIL>"]
TURKISH_CHARS = list("abcçdefgğhıijklmnoöprsştuüvyz")
PUNCTUATIONS = list(".,!? -:;")
VOCAB = SPECIAL_TOKENS + TURKISH_CHARS + PUNCTUATIONS
CHAR_TO_ID = {ch: idx for idx, ch in enumerate(VOCAB)}
ID_TO_CHAR = {idx: ch for idx, ch in enumerate(VOCAB)}

def normalize_turkish_text(text: str) -> str:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    return re.sub(r"\s+", " ", text).strip()

def text_to_sequence(text: str) -> list[int]:
    text = normalize_turkish_text(text)
    seq = [CHAR_TO_ID["<BOS>"]]
    for ch in text:
        if ch in CHAR_TO_ID:
            seq.append(CHAR_TO_ID[ch])
        elif ch == " ":
            seq.append(CHAR_TO_ID.get(" ", CHAR_TO_ID["<SIL>"]))
        else:
            seq.append(CHAR_TO_ID["<UNK>"])
    seq.append(CHAR_TO_ID["<EOS>"])
    return seq

print(f"Vocab hazır! Boyut: {len(VOCAB)}")

In [ ]:
# 3. Audio DSP (Mel-Spektrogram & Griffin-Lim Çıkarıcı)
class AudioConfig:
    sample_rate: int = 22050
    n_fft: int = 1024
    win_length: int = 1024
    hop_length: int = 256
    n_mels: int = 80
    f_min: float = 0.0
    f_max: float = 8000.0

class AudioProcessor(nn.Module):
    def __init__(self, config: AudioConfig = AudioConfig()):
        super().__init__()
        self.config = config
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.sample_rate,
            n_fft=config.n_fft,
            win_length=config.win_length,
            hop_length=config.hop_length,
            f_min=config.f_min,
            f_max=config.f_max,
            n_mels=config.n_mels,
            power=1.0,
        )
        self.griffin_lim = T.GriffinLim(
            n_fft=config.n_fft,
            win_length=config.win_length,
            hop_length=config.hop_length,
            n_iter=32,
            power=1.0
        )
        mel_fb = torchaudio.functional.melscale_fbanks(
            n_freqs=(config.n_fft // 2) + 1,
            f_min=config.f_min,
            f_max=config.f_max,
            n_mels=config.n_mels,
            sample_rate=config.sample_rate,
        )
        self.register_buffer("inv_mel_fb", torch.pinverse(mel_fb).t())

    def wav_to_mel(self, wav: torch.Tensor) -> torch.Tensor:
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        mel = self.mel_transform(wav)
        return torch.log(torch.clamp(mel, min=1e-5))

    def mel_to_wav(self, log_mel: torch.Tensor) -> torch.Tensor:
        if log_mel.dim() == 2:
            log_mel = log_mel.unsqueeze(0)
        mel = torch.exp(log_mel)
        linear_spec = torch.matmul(self.inv_mel_fb, mel)
        linear_spec = torch.clamp(linear_spec, min=0.0)
        return self.griffin_lim(linear_spec).squeeze(0)

dsp = AudioProcessor()
print("Audio DSP hazır!")

In [ ]:
# 4. FastSpeech 2 Model Mimarisi (From Scratch)
class FFTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, conv_filter_size: int, dropout: float = 0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.conv1 = nn.Conv1d(d_model, conv_filter_size, kernel_size=9, padding=4)
        self.conv2 = nn.Conv1d(conv_filter_size, d_model, kernel_size=1)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(residual + self.dropout(attn_out))
        residual = x
        out = F.relu(self.conv1(x.transpose(1, 2)))
        out = self.conv2(out).transpose(1, 2)
        return self.norm2(residual + self.dropout(out))

class VariancePredictor(nn.Module):
    def __init__(self, d_model: int, filter_size: int = 256, kernel_size: int = 3, dropout: float = 0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(d_model, filter_size, kernel_size=kernel_size, padding=kernel_size // 2)
        self.norm1 = nn.LayerNorm(filter_size)
        self.conv2 = nn.Conv1d(filter_size, filter_size, kernel_size=kernel_size, padding=kernel_size // 2)
        self.norm2 = nn.LayerNorm(filter_size)
        self.linear = nn.Linear(filter_size, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.norm1(self.conv1(x.transpose(1, 2)).transpose(1, 2)))
        out = self.dropout(out)
        out = F.relu(self.norm2(self.conv2(out.transpose(1, 2)).transpose(1, 2)))
        out = self.dropout(out)
        return self.linear(out).squeeze(-1)

class LengthRegulator(nn.Module):
    def forward(self, x, durations):
        output = []
        for batch_idx in range(x.size(0)):
            seq = x[batch_idx]
            durs = durations[batch_idx]
            expanded = []
            for token_vec, dur in zip(seq, durs):
                dur_int = max(1, int(round(dur.item())))
                expanded.append(token_vec.repeat(dur_int, 1))
            output.append(torch.cat(expanded, dim=0))
        max_len = max(o.size(0) for o in output)
        padded_output = [F.pad(o, (0, 0, 0, max_len - o.size(0))) if max_len > o.size(0) else o for o in output]
        return torch.stack(padded_output)

class DorinaTTSModel(nn.Module):
    def __init__(self, vocab_size: int = 42, d_model: int = 192, n_heads: int = 2, layers: int = 3, n_mels: int = 80):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.encoder = nn.ModuleList([FFTBlock(d_model, n_heads, 384) for _ in range(layers)])
        self.dur_pred = VariancePredictor(d_model)
        self.pitch_pred = VariancePredictor(d_model)
        self.energy_pred = VariancePredictor(d_model)
        self.pitch_emb = nn.Conv1d(1, d_model, kernel_size=9, padding=4)
        self.energy_emb = nn.Conv1d(1, d_model, kernel_size=9, padding=4)
        self.lr = LengthRegulator()
        self.decoder = nn.ModuleList([FFTBlock(d_model, n_heads, 384) for _ in range(layers)])
        self.mel_linear = nn.Linear(d_model, n_mels)

    def forward(self, text_seq, dur=None, pitch=None, energy=None):
        x = self.embedding(text_seq)
        for l in self.encoder: x = l(x)
        p_d, p_p, p_e = self.dur_pred(x), self.pitch_pred(x), self.energy_pred(x)
        d = dur if dur is not None else torch.clamp(torch.round(F.softplus(p_d) * 3 + 1), min=1, max=12)
        p = pitch if pitch is not None else p_p
        e = energy if energy is not None else p_e
        x = x + self.pitch_emb(p.unsqueeze(1)).transpose(1, 2) + self.energy_emb(e.unsqueeze(1)).transpose(1, 2)
        x = self.lr(x, d)
        for l in self.decoder: x = l(x)
        mel = self.mel_linear(x).transpose(1, 2)
        return mel, p_d, p_p, p_e

print("Model mimarisi tanımlandı!")

In [ ]:
# 5. Gerçek Türkçe Veri Kümesini İndir (Bentropi Kadın Sesi)
import io
os.makedirs("dataset/wavs", exist_ok=True)
url = "https://huggingface.co/datasets/falan42/Turkish-Bentropi-tts-dataset/resolve/main/data/train-00000-of-00001.parquet"
df = pd.read_parquet(url)

metadata = []
for i, row in df.iterrows():
    text = str(row.get('transcription') or row.get('text') or '').strip()
    if not text or len(text) < 5: continue
    audio_bytes = row['audio']['bytes']
    wav_data, sr = sf.read(io.BytesIO(audio_bytes))
    w_tensor = torch.tensor(wav_data, dtype=torch.float32)
    if w_tensor.dim() == 1: w_tensor = w_tensor.unsqueeze(0)
    else: w_tensor = w_tensor.t()[:1]
    if sr != 22050:
        w_tensor = torchaudio.transforms.Resample(sr, 22050)(w_tensor)
    fn = f"dorina_{i:04d}.wav"
    sf.write(f"dataset/wavs/{fn}", w_tensor.squeeze(0).numpy(), 22050)
    metadata.append(f"{fn}|{text}")

with open("dataset/metadata.csv", "w", encoding="utf-8") as f:
    f.write("\n".join(metadata))

print(f"Toplam {len(metadata)} adet ses kaydı hazırlandı!")

In [ ]:
# 6. Modeli GPU Üzerinde Eğit (Training)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DorinaTTSModel().to(device)
dsp = AudioProcessor().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Dataset hazırlığı
samples = []
for line in metadata:
    fn, text = line.split("|")
    w_np, _ = sf.read(f"dataset/wavs/{fn}")
    with torch.no_grad():
        mel = dsp.wav_to_mel(torch.tensor(w_np, dtype=torch.float32).to(device)).squeeze(0)
    seq = torch.tensor(text_to_sequence(text), dtype=torch.long)
    samples.append((seq, mel))

print(f"[*] {len(samples)} örnek ile eğitim başlatılıyor (Cihaz: {device})...")

EPOCHS = 100
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for seq, mel in samples:
        seq = seq.unsqueeze(0).to(device)
        mel = mel.unsqueeze(0).to(device)
        dur = torch.full((1, seq.shape[-1]), fill_value=max(1, mel.shape[-1] // seq.shape[-1]), dtype=torch.float, device=device)
        diff = mel.shape[-1] - int(dur.sum().item())
        dur[0, -1] += diff
        p = torch.randn(1, seq.shape[-1], device=device)
        e = torch.randn(1, seq.shape[-1], device=device)
        
        optimizer.zero_grad()
        p_mel, p_dur, p_p, p_e = model(seq, dur, p, e)
        min_t = min(p_mel.shape[-1], mel.shape[-1])
        loss = F.l1_loss(p_mel[:, :, :min_t], mel[:, :, :min_t]) * 2.0 + F.mse_loss(p_dur, dur)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{EPOCHS}] - Loss: {epoch_loss / len(samples):.4f}")

torch.save(model.state_dict(), "dorina_tts_final.pt")
print("Model eğitimi tamamlandı ve kaydedildi!")

In [ ]:
# 7. Dorina'ya İstediğin Metni Konuştur ve Dinle!
def dorina_speak(text: str):
    model.eval()
    with torch.no_grad():
        seq = torch.tensor([text_to_sequence(text)], dtype=torch.long, device=device)
        mel_out, _, _, _ = model(seq)
        wav = dsp.mel_to_wav(mel_out.squeeze(0))
        wav_np = wav.cpu().numpy()
        sf.write("output.wav", wav_np, 22050)
        display(Audio("output.wav", rate=22050))

# Denemek istediğin metni yaz:
dorina_speak("Selam Talha, ben Dorina!")